# SmartCrop — Notebook 06: Model Evaluation

Metrics are MAE (t/ha), RMSE (t/ha), and R². This notebook does not report classification accuracy.


## 1. Introduction

Evaluation asks two questions. Which candidate should be kept, using only the validation season? Once that choice is fixed, what error does it make on later seasons that it has not seen?


## 2. Objectives

1. Select the model with the best validation R².
2. Score that model on the held-out seasons.
3. Compare the test R² with an 80% target defined as R² >= 0.80.
4. Plot actual against predicted, residuals, and the error distribution.
5. Rank inputs with permutation importance and list the top five.


## 3. Dataset Description

The same clean synthetic file and temporal split as Notebook 05. Importance is computed on the validation season so the test season is used for the error summary rather than for explanation.


## 4. Methodology

The academic 80% target, where a project statement asks for it, is defined here as test-set R² of at least 0.80. R² is the share of yield variance the predictions track. It is not the percentage of farms predicted exactly, and it is not classification accuracy.

Permutation importance shuffles one raw input at a time and records the drop in validation R². A large drop means the fitted model was using that input. It does not mean the input caused yield in Macheke.


## 5. Implementation


In [1]:
from pathlib import Path
import sys
import pandas as pd

def find_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    if (start / "data-science").is_dir():
        candidates.insert(0, start / "data-science")
    for candidate in candidates:
        if (candidate / "src" / "data" / "schema.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the SmartCrop repository.")

ROOT = find_root()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 120)

from data.schema import SYNTHETIC_DISCLAIMER
from evaluation.metrics import regression_metrics
from models.training import (
    PERFORMANCE_TARGET_R2,
    RANDOM_SEED,
    evaluate_split,
    fit_final_model,
    permutation_feature_importance,
    run_model_comparison,
    select_model_name,
)
from reporting.model_report import prediction_frame, save_evaluation_figures

print(SYNTHETIC_DISCLAIMER)
clean = pd.read_csv(ROOT / "data" / "processed" / "clean_dataset.csv")
comparison, _details = run_model_comparison(clean, RANDOM_SEED)
chosen = select_model_name(comparison)
model, masks = fit_final_model(clean, chosen, RANDOM_SEED)
validation = clean.loc[clean["year"].isin(masks["validation_years"])]
test = clean.loc[clean["year"].isin(masks["test_years"])]
validation_metrics = evaluate_split(model, validation)
test_metrics = evaluate_split(model, test)
importance = permutation_feature_importance(model, validation, RANDOM_SEED)
evaluated = prediction_frame(model, test)
print(f"Selected on validation R²: {chosen}")
print(f"Validation MAE {validation_metrics['mae']:.3f}, RMSE {validation_metrics['rmse']:.3f}, R² {validation_metrics['r2']:.3f}")
print(f"Test MAE {test_metrics['mae']:.3f}, RMSE {test_metrics['rmse']:.3f}, R² {test_metrics['r2']:.3f}")
print(f"80% target means test R² >= {PERFORMANCE_TARGET_R2:.2f}. Met: {test_metrics['r2'] >= PERFORMANCE_TARGET_R2}")
display(importance)


SYNTHETIC DATA. These rows were generated by the SmartCrop synthetic generator for software and model-development exercises. They are not measurements from Macheke, Mashonaland East, Zimbabwe, and they must not be cited as agricultural statistics.


Selected on validation R²: Random Forest Regressor
Validation MAE 0.285, RMSE 0.352, R² -0.310
Test MAE 0.264, RMSE 0.342, R² 0.468
80% target means test R² >= 0.80. Met: False


,rank,feature,importance_mean,importance_std
0,1,fertilizer_kg_ha,0.487800,0.094490
1,2,rainfall_mm,0.155712,0.035686
2,3,planting_day,0.064149,0.021657
3,4,soil_nitrogen,-0.001214,0.006352
4,5,avg_temperature_c,-0.003086,0.002643
5,6,soil_ph,-0.003847,0.008687
6,7,soil_organic_carbon,-0.013674,0.012917
7,8,min_temperature_c,-0.021125,0.007055
8,9,max_temperature_c,-0.029573,0.014255


In [2]:
reports = ROOT / "reports" / "model_reports"
reports.mkdir(parents=True, exist_ok=True)
metrics = pd.DataFrame([
    {"split": "validation", "mae": validation_metrics["mae"], "rmse": validation_metrics["rmse"], "r2": validation_metrics["r2"]},
    {"split": "test", "mae": test_metrics["mae"], "rmse": test_metrics["rmse"], "r2": test_metrics["r2"]},
])
metrics.to_csv(reports / "model_metrics.csv", index=False)
comparison.to_excel(reports / "model_comparison.xlsx", index=False)
importance.to_csv(reports / "feature_importance.csv", index=False)
evaluated.to_csv(reports / "sample_predictions.csv", index=False)
save_evaluation_figures(evaluated, importance, ROOT / "reports" / "figures")
print("Wrote actual_vs_predicted, residuals, prediction_error_distribution, and feature_importance figures.")
display(metrics.round(4))
display(evaluated.head())


Wrote actual_vs_predicted, residuals, prediction_error_distribution, and feature_importance figures.


,split,mae,rmse,r2
0,validation,0.2850,0.3516,-0.3097
1,test,0.2644,0.3422,0.4675


,record_id,season,year,ward,yield_t_ha,predicted_yield_t_ha,residual_t_ha
703,SC-SYN-000723,2022/2023,2023,SYN-W01,2.099,2.076,0.023
704,SC-SYN-000734,2022/2023,2023,SYN-W01,1.988,2.036,-0.048
705,SC-SYN-000739,2022/2023,2023,SYN-W01,1.866,1.627,0.239
706,SC-SYN-000748,2022/2023,2023,SYN-W01,2.118,2.077,0.041
707,SC-SYN-000752,2022/2023,2023,SYN-W01,2.008,2.010,-0.002


## 6. Results

Read the printed test MAE, RMSE, and R². The target line states whether test R² reached 0.80. The importance table is ranked. Near-zero or negative importance means shuffling that input did not reduce validation R².


## 7. Visualizations

`actual_vs_predicted.png` plots the test seasons against the equality line. `residuals.png` plots error against the prediction. `prediction_error_distribution.png` is the histogram of those errors, in tonnes per hectare. `feature_importance.png` shows the five highest permutation scores.


## 8. Interpretation

Judge the model by tonnes per hectare of error, not by an accuracy percentage. If R² is below 0.80, the model does not meet the stated target on these held-out synthetic seasons. Feature importance indicates variables that contributed strongly to model predictions. It does not prove causation, and the generator itself planted relationships for rainfall, fertilizer, and planting day.


## 9. Conclusion

The selected algorithm and its test metrics are fixed before serialization. Notebook 07 saves the pipeline and the metadata the API will load.
